# vllm-serving-lab on Colab

Runtime > Change runtime type > **T4 GPU** before starting.

Put `vllm-serving-lab.zip` in the top level of your Google Drive (My Drive).

Results, golden outputs and logs are stored on Drive, so nothing is lost if the session disconnects.
After a disconnect, run the **Setup** cells again and continue from the step you were on.

## Setup (run every session)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf vllm-serving-lab && unzip -q -o /content/drive/MyDrive/vllm-serving-lab.zip
%cd /content/vllm-serving-lab
!chmod +x *.sh

# Keep outputs on Drive
!mkdir -p /content/drive/MyDrive/vllm-lab/results /content/drive/MyDrive/vllm-lab/golden /content/drive/MyDrive/vllm-lab/logs
!ln -sfn /content/drive/MyDrive/vllm-lab/results results
!ln -sfn /content/drive/MyDrive/vllm-lab/golden golden
!ln -sfn /content/drive/MyDrive/vllm-lab/logs logs

In [ ]:
!pip install -q -r requirements.txt

If the next cell fails with an import error, use Runtime > Restart session, then run the Setup cells again (skip the pip install).

In [ ]:
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 1. Serve the model

The first start downloads the model and takes a few minutes.

In [ ]:
!./serve.sh baseline

In [ ]:
!curl -s http://localhost:8000/v1/completions -H "Content-Type: application/json" \
  -d '{"model": "Qwen/Qwen2.5-1.5B-Instruct", "prompt": "The capital of France is", "max_tokens": 16, "temperature": 0}'

In [ ]:
# KV cache and block info from the startup log
!grep -iE "KV cache|GPU blocks|concurrency" logs/server_baseline.log | head -20

## 2. Static vs continuous batching

Batch size 1 uses fewer requests, otherwise it takes over an hour on a T4.

In [ ]:
!python bench_batching.py --mode static --batch-size 1 --num-requests 64
!python bench_batching.py --mode continuous --batch-size 1 --num-requests 64

In [ ]:
for b in (8, 32, 64):
    for mode in ("static", "continuous"):
        !python bench_batching.py --mode {mode} --batch-size {b} --num-requests 512

In [ ]:
# Control: equal output lengths
!python bench_batching.py --mode static --batch-size 32 --num-requests 512 --fixed-output-len 256
!python bench_batching.py --mode continuous --batch-size 32 --num-requests 512 --fixed-output-len 256

## 3. Prefix caching

In [ ]:
!./serve.sh baseline && ./run_benchmarks.sh prefix baseline

In [ ]:
!./serve.sh prefix && ./run_benchmarks.sh prefix prefix

## 4. Speculative decoding

If `spec-draft` runs out of memory, open `serve.sh` and change `--max-model-len 8192` to `4096`,
then rerun the baseline and spec-ngram cells too so all three use the same setting.

In [ ]:
!./serve.sh baseline && ./run_benchmarks.sh spec baseline

In [ ]:
!./serve.sh spec-ngram && ./run_benchmarks.sh spec spec-ngram

In [ ]:
!./serve.sh spec-draft && ./run_benchmarks.sh spec spec-draft

In [ ]:
!cat results/spec_metrics_spec-ngram.txt results/spec_metrics_spec-draft.txt

## 5. Accuracy regression

The golden outputs must come from the baseline config.

In [ ]:
!./serve.sh baseline && python make_golden.py

In [ ]:
!./serve.sh baseline && pytest -s tests   # sanity check: baseline vs itself

In [ ]:
!./serve.sh prefix && pytest -s tests

In [ ]:
!./serve.sh spec-ngram && pytest -s tests

In [ ]:
!./serve.sh spec-draft && pytest -s tests

## 6. Summary

In [ ]:
!python summarize.py > results/RESULTS.md
!cat results/RESULTS.md

In [ ]:
!pkill -f 'vllm serve' || true   # stop the server when done